# Cuaderno de clase
## Mecánica Celeste (2026-1) con Jorge I. Zuluaga
## El problema de los dos cuerpos en el tiempo

In [107]:
import pymcel as pc
import numpy as np
deg = np.pi / 180

In [108]:
e	= 0.1911663355386932
a	= 0.9223803173917017 * pc.constantes.au
q	= 0.7460522521429133 * pc.constantes.au
i	= 3.340958441017069*deg
node = 203.8996515621043*deg
peri = 126.6728325163065*deg
#M = 312.8054663584516*deg
tp = 2461042.918242006079 * 86400 # fecha juliana
mu = pc.constantes.mu_sun + pc.constantes.mu_jupiter + pc.constantes.mu_saturn# Este es el mu de todo el sistema solar

Varaibles derivadas:

In [109]:
h = np.sqrt(mu * a * (1 - e**2))
b = a * np.sqrt(1 - e**2)

h, b

(np.float64(4202992298266956.0), np.float64(135441343761.16716))

¿Cuál es el tiempo?

In [110]:
from astropy.time import Time

In [111]:
t = Time("2029-04-13 18:52:00", scale='tdb').jd * 86400
t

np.float64(212737560720.0)

Escribir la ecuación en la forma: f(E) = 0

In [112]:
def ecuacion_kepler(E):
  funcion = E - e*np.sin(E) - h/(a*b)*(t-tp)
  return funcion

Resolver la ecuación de Kepler:

In [113]:
from scipy.optimize import newton

In [114]:
E = newton(ecuacion_kepler, 0)
E

np.float64(23.094856506562053)

Fórmulas aproximadas:

1. $E = M + e \sin M$
2. $E = M + e \sin M + e^2/2 \sin 2M $

In [115]:
e

0.1911663355386932

In [116]:
n = h/(a*b)
M = n*(t-tp)

E = M + e*np.sin(M)
print(f"Anomalía excéntrica a primer orden: {E}")
E = M + e*np.sin(M) + e**2/2 * np.sin(2*M)
print(f"Anomalía excéntrica a segundo orden: {E}")
E = M + (e-1/3*e**3)*np.sin(M) + e**2/2 * np.sin(2*M) + 3/8*e**3*np.sin(3*M)
print(f"Anomalía excéntrica a tercer orden: {E}")

Anomalía excéntrica a primer orden: 23.082715289812768
Anomalía excéntrica a segundo orden: 23.092923725110193
Anomalía excéntrica a tercer orden: 23.096801434558984


In [117]:
pc.kepler_newton(M, e, delta=1e-15)

(np.float64(23.09485650656205), np.float64(0.0), 6)

Obtenemos ahora la anomalía verdadera:

In [118]:
f = 2*np.arctan(np.sqrt((1+e)/(1-e))*np.tan(E/2))
f

np.float64(-2.200858687003166)

Una vez tenemos la anomalía verdadera podemos calcular la posición en su sistema perifocal:

In [119]:
p = a*(1-e**2)
r = p / (1 + e * np.cos(f))
xf = r * np.cos(f)
yf = r * np.sin(f)
zf = 0

Usando la rotación en el espacio:

In [120]:
import spiceypy as spy
R = spy.eul2m(-node, -i, -peri, 3, 1, 3)

r = R @ np.array([xf, yf, zf])
r

array([-1.36359471e+11, -6.20609716e+10,  8.73030986e+07])

Vamos a comparar la posición con la que nos da consulta horizons:


In [121]:
tabla, jd, X = pc.consulta_horizons(id='Apophis', location='@SSB', epochs='2029-04-13 18:52:00')
X[:3]

c:\Users\juane\AppData\Local\Programs\Python\Python312\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


array([-1.37263651e+11, -6.04834602e+10, -4.52634041e+06])

# Rutinas de spyceypy

Tres rutinas: 

1. Determinación de órbitas: Dado r y v, ¿cuáles son los elementos orbitales?

`spy.oscelt` (osculating elements).

2. Determinación de estado: Dados los elementos orbitales, ¿cuál es el estado?

 `spy.conics`.

3. Formalismo f y g para la determinación del vector de estado. 

 `spy.prop2b`

In [123]:
M = 312.8054663584516*deg
spy.conics([q, e, i, node, peri, M, tp, mu], t)

array([-1.59199763e+11,  3.20415674e+10, -5.47527296e+09, -3.30865873e+03,
       -2.56899524e+04,  1.29285851e+03])